# Base C — Calidad de datos

**Proyecto Final — Licenciatura en Ciencia de Datos**

Objetivo: validar **Base C** antes de cualquier modelado.

### Alcance
- inventario de artefactos y runs;
- estructura y grano;
- calidad estructural;
- cobertura temporal;
- reconciliación `raw → empresa-entidad-mes → empresa-mes`;
- consistencia con la muestra empresarial;
- diagnóstico de `monto = 0` y su relación con `n_entidades`.

### Fuera de alcance
- modelado;
- selección de features;
- optimización o redefinición del target;
- análisis predictivo;
- cruces con Base A o Base B.

### Regla de trabajo
Este notebook **no imputa, elimina ni corrige automáticamente** observaciones.

Cada hallazgo debe distinguir, cuando corresponda, entre:
- **ERROR DE DATOS**
- **AUSENCIA LEGÍTIMA DE DEUDA**
- **AUSENCIA DE INFORMACIÓN**
- **COMPORTAMIENTO CREDITICIO REAL**
- **DECISIÓN METODOLÓGICA**

> El target exploratorio existente se conserva en los archivos, pero su definición y calidad se analizarán **post-EDA**, al diseñar el modelo.

Esta auditoría final corresponde a la corrida histórica `20260910_234243_324893`. Las nuevas adquisiciones generan únicamente insumos y cobertura; su calidad operativa se verifica con el contrato relacional y `docs/validar_relacional.py`.


In [5]:
from pathlib import Path
import json
import re

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.width", 180)

# -------------------------------------------------------------------
# CONFIGURACIÓN
# -------------------------------------------------------------------
RUN_ID = "20260910_234243_324893"  # Auditoría final histórica; no es el selector operativo.

# data_c está ubicada un nivel por encima de la carpeta del notebook.
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "data_c").is_dir()), Path.cwd())
DATA_C_PATH = ROOT / "data_c"

if not DATA_C_PATH.is_dir():
    raise FileNotFoundError(
        f"No encontré la carpeta data_c en: {DATA_C_PATH}"
    )

print("DATA_C:", DATA_C_PATH)

DATA_C: C:\Users\Asus\OneDrive\BRIAN\CAECE\4°año_2°Cuatri\PROYECTO FINAL EN CIENCIA DE DATOS\scripts\data_c


## 1. Inventario de artefactos y selección del run

El notebook detecta todos los `run_id` disponibles.  
Si `RUN_ID = None`, selecciona automáticamente el **run completo con mayor cantidad de empresas input**; ante empate, el más reciente.


In [6]:
members = [
    p.relative_to(DATA_C_PATH).as_posix()
    for p in DATA_C_PATH.rglob("*")
    if p.is_file()
]

def read_csv_z(member, **kwargs):
    return pd.read_csv(DATA_C_PATH / member, **kwargs)

def read_json_z(member):
    with (DATA_C_PATH / member).open("r", encoding="utf-8") as f:
        return json.load(f)

def exists(member):
    return (DATA_C_PATH / member).is_file()

# Detectar prefijos de runs relativos a data_c.
run_pat = re.compile(r"^(runs/\d{4}-\d{2}-\d{2}/([^/]+))/")
run_prefixes = {}

for m in members:
    mt = run_pat.match(m)
    if mt:
        run_prefixes[mt.group(2)] = mt.group(1)

required_processed = [
    "base_c_empresas_input.csv",
    "coverage.csv",
    "base_c_empresa_entidad_mes.csv",
    "base_c_empresa_mes.csv",
    "base_c_empresa_subsector_input.csv",
    "base_c_empresa_subsector_mes.csv",
    "base_c_empresas_enriquecida.csv",
]

inventory_rows = []
for rid, prefix in sorted(run_prefixes.items()):
    proc = f"{prefix}/processed/"
    present = {file: exists(proc + file) for file in required_processed}
    complete = all(present.values())

    def nrows_if(file):
        if exists(proc + file):
            return len(read_csv_z(proc + file, dtype=str))
        return np.nan

    raw_json = [
        m for m in members
        if m.startswith(f"{prefix}/raw/deudas/") and m.endswith(".json")
    ]

    inventory_rows.append({
        "run_id": rid,
        "complete_processed": complete,
        "empresas_input": nrows_if("base_c_empresas_input.csv"),
        "coverage_rows": nrows_if("coverage.csv"),
        "long_rows": nrows_if("base_c_empresa_entidad_mes.csv"),
        "panel_rows": nrows_if("base_c_empresa_mes.csv"),
        "raw_json": len(raw_json),
        "run_summary": exists(f"{prefix}/run_summary.md"),
        "run_metadata": exists(f"{prefix}/run_metadata.json"),
    })

run_inventory = pd.DataFrame(inventory_rows)
display(run_inventory)

if RUN_ID is None:
    candidates_run = run_inventory[run_inventory["complete_processed"]].copy()
    if candidates_run.empty:
        raise RuntimeError("No hay ningún run con todos los artefactos processed requeridos.")
    candidates_run["empresas_input_num"] = pd.to_numeric(candidates_run["empresas_input"])
    selected = candidates_run.sort_values(
        ["empresas_input_num", "run_id"], ascending=[False, False]
    ).iloc[0]
    RUN_ID_SELECTED = selected["run_id"]
else:
    if RUN_ID not in run_prefixes:
        raise ValueError(f"RUN_ID no encontrado: {RUN_ID}")
    RUN_ID_SELECTED = RUN_ID

RUN_PREFIX = run_prefixes[RUN_ID_SELECTED]
PROC_PREFIX = f"{RUN_PREFIX}/processed/"

display(Markdown(f"### Run seleccionado: `{RUN_ID_SELECTED}`"))
print(f"Archivos detectados: {len(members):,}")

,run_id,complete_processed,empresas_input,coverage_rows,long_rows,panel_rows,raw_json,run_summary,run_metadata
0,20260910_233743_640942,True,10,10,3864,240,10,True,True
1,20260910_234243_324893,True,100,100,28968,2400,100,True,True


### Run seleccionado: `20260910_234243_324893`

Archivos detectados: 131


In [7]:
# Manifest global, si existe
manifest_member = "run_manifest.csv"
if exists(manifest_member):
    run_manifest = read_csv_z(manifest_member, dtype=str)
    display(run_manifest)
else:
    run_manifest = pd.DataFrame()
    print(f"No se encontró el manifest: {DATA_C_PATH / manifest_member}")

,run_id,run_date,run_at,input_file,companies_input,stopped_for_maintenance,api,amount_unit,immutable_run_path,companies_attempted,companies_ok,companies_no_data,long_rows,panel_rows
0,20260910_233743_640942,2026-09-10,2026-09-10T23:37:43.640942-03:00,C:\Users\Asus\OneDrive\BRIAN\CAECE\4°año_2°Cua...,10,False,BCRA Central de Deudores / Deudas Historicas,miles de pesos,C:\Users\Asus\OneDrive\BRIAN\CAECE\4°año_2°Cua...,10,10,0,3864,240
1,20260910_234243_324893,2026-09-10,2026-09-10T23:42:43.324893-03:00,C:\Users\Asus\OneDrive\BRIAN\CAECE\4°año_2°Cua...,100,False,BCRA Central de Deudores / Deudas Historicas,miles de pesos,C:\Users\Asus\OneDrive\BRIAN\CAECE\4°año_2°Cua...,100,100,0,28968,2400


## 2. Carga de tablas y estructura

Se cargan únicamente los artefactos del run seleccionado.  
La muestra empresarial original se mantiene separada de la respuesta crediticia.


In [8]:
empresas = read_csv_z(PROC_PREFIX + "base_c_empresas_input.csv", dtype=str)
coverage = read_csv_z(PROC_PREFIX + "coverage.csv", dtype=str)
long = read_csv_z(PROC_PREFIX + "base_c_empresa_entidad_mes.csv", dtype=str)
panel = read_csv_z(PROC_PREFIX + "base_c_empresa_mes.csv", dtype=str)
empresa_subsector = read_csv_z(PROC_PREFIX + "base_c_empresa_subsector_input.csv", dtype=str)
panel_subsector = read_csv_z(PROC_PREFIX + "base_c_empresa_subsector_mes.csv", dtype=str)
empresas_enriquecida = read_csv_z(PROC_PREFIX + "base_c_empresas_enriquecida.csv", dtype=str)

tables = {
    "empresas_input": empresas,
    "coverage": coverage,
    "empresa_entidad_mes": long,
    "empresa_mes": panel,
    "empresa_subsector_input": empresa_subsector,
    "empresa_subsector_mes": panel_subsector,
    "empresas_enriquecida": empresas_enriquecida,
}

shape_summary = pd.DataFrame([
    {"tabla": name, "filas": len(df), "columnas": len(df.columns)}
    for name, df in tables.items()
])
display(shape_summary)

for name, df in tables.items():
    print(f"\n{name}:")
    print(list(df.columns))


,tabla,filas,columnas
0,empresas_input,100,14
1,coverage,100,9
2,empresa_entidad_mes,28968,14
3,empresa_mes,2400,31
4,empresa_subsector_input,102,6
5,empresa_subsector_mes,2448,33
6,empresas_enriquecida,100,23



empresas_input:
['empresa_id', 'rubro', 'razon_social', 'cuit', 'cuit_normalizado', 'subsectores', 'perfil_sectorial', 'grupo_economico', 'estado_seleccion', 'fuente_consolidacion', 'documentos_soporte', 'cuit_formato_valido', 'observaciones', 'cuit_check_digit_valido']

coverage:
['empresa_id', 'cuit_normalizado', 'razon_social', 'rubro', 'subsectores', 'attempted', 'http_status', 'outcome', 'message']

empresa_entidad_mes:
['empresa_id', 'cuit_normalizado', 'razon_social', 'denominacion_bcra', 'rubro', 'subsectores', 'perfil_sectorial', 'grupo_economico', 'periodo', 'entidad', 'situacion', 'monto_miles_pesos', 'en_revision', 'proceso_judicial']

empresa_mes:
['empresa_id', 'cuit_normalizado', 'razon_social', 'denominacion_bcra', 'rubro', 'subsectores', 'perfil_sectorial', 'grupo_economico', 'periodo', 'deuda_total_miles', 'deuda_promedio_entidad_miles', 'deuda_mediana_entidad_miles', 'deuda_max_entidad_miles', 'n_entidades', 'situacion_max', 'situacion_promedio', 'en_revision_any', 

In [9]:
# Conversión auxiliar para EDA; NO modifica los archivos fuente.
long_num = long.copy()
panel_num = panel.copy()

for c in ["monto_miles_pesos", "situacion"]:
    if c in long_num:
        long_num[c] = pd.to_numeric(long_num[c], errors="coerce")

numeric_panel_cols = [
    "deuda_total_miles", "deuda_promedio_entidad_miles",
    "deuda_mediana_entidad_miles", "deuda_max_entidad_miles",
    "n_entidades", "situacion_max", "situacion_promedio",
    "hhi_concentracion_deuda", "top1_share_deuda", "top3_share_deuda",
    "deuda_lag_1m", "deuda_var_abs_1m_miles", "deuda_var_pct_1m",
    "n_entidades_lag_1m", "n_entidades_delta_1m",
    "situacion_lag_1m", "situacion_delta_1m", "situacion_next_1m",
]
for c in numeric_panel_cols:
    if c in panel_num:
        panel_num[c] = pd.to_numeric(panel_num[c], errors="coerce")

long_num["periodo_dt"] = pd.to_datetime(
    long_num["periodo"].astype(str) + "01",
    format="%Y%m%d",
    errors="coerce",
)
panel_num["periodo_dt_eda"] = pd.to_datetime(
    panel_num["periodo"].astype(str) + "01",
    format="%Y%m%d",
    errors="coerce",
)


### Grano esperado

| Tabla | Grano / clave esperada |
|---|---|
| `empresas_input` | una persona jurídica: `cuit_normalizado` |
| `coverage` | una consulta por CUIT dentro del run: `cuit_normalizado` |
| `empresa_entidad_mes` | `cuit_normalizado + periodo + entidad` |
| `empresa_mes` | `cuit_normalizado + periodo` |
| `empresa_subsector_input` | `cuit_normalizado + subsector` |
| `empresa_subsector_mes` | `cuit_normalizado + periodo + subsector` |
| `empresas_enriquecida` | `cuit_normalizado` |


In [10]:
key_specs = {
    "empresas_input": ["cuit_normalizado"],
    "coverage": ["cuit_normalizado"],
    "empresa_entidad_mes": ["cuit_normalizado", "periodo", "entidad"],
    "empresa_mes": ["cuit_normalizado", "periodo"],
    "empresa_subsector_input": ["cuit_normalizado", "subsector"],
    "empresa_subsector_mes": ["cuit_normalizado", "periodo", "subsector"],
    "empresas_enriquecida": ["cuit_normalizado"],
}

grain_rows = []
for name, keys in key_specs.items():
    df = tables[name]
    dup = int(df.duplicated(keys, keep=False).sum())
    grain_rows.append({
        "tabla": name,
        "filas": len(df),
        "clave": " + ".join(keys),
        "claves_unicas": df[keys].drop_duplicates().shape[0],
        "filas_en_claves_duplicadas": dup,
        "estado": "PASS" if dup == 0 else "REVISAR",
    })
grain_check = pd.DataFrame(grain_rows)
display(grain_check)


,tabla,filas,clave,claves_unicas,filas_en_claves_duplicadas,estado
0,empresas_input,100,cuit_normalizado,100,0,PASS
1,coverage,100,cuit_normalizado,100,0,PASS
2,empresa_entidad_mes,28968,cuit_normalizado + periodo + entidad,28968,0,PASS
3,empresa_mes,2400,cuit_normalizado + periodo,2400,0,PASS
4,empresa_subsector_input,102,cuit_normalizado + subsector,102,0,PASS
5,empresa_subsector_mes,2448,cuit_normalizado + periodo + subsector,2448,0,PASS
6,empresas_enriquecida,100,cuit_normalizado,100,0,PASS


## 3. Cobertura de consultas BCRA

**Regla de interpretación en este EDA:**
- `HTTP 200 / outcome=ok`: información técnica disponible.
- `404 / no_data`: **AUSENCIA DE INFORMACIÓN en Central de Deudores**, no se interpreta automáticamente como deuda cero.
- errores, mantenimiento o consultas no realizadas: **AUSENCIA DE INFORMACIÓN**.
- no se reemplaza ni elimina ninguna empresa por el resultado crediticio.


In [11]:
coverage_summary = (
    coverage.groupby(["attempted", "http_status", "outcome"], dropna=False)
    .size()
    .reset_index(name="empresas")
    .sort_values("empresas", ascending=False)
)
display(coverage_summary)

input_cuits = set(empresas["cuit_normalizado"].astype(str))
coverage_cuits = set(coverage["cuit_normalizado"].astype(str))
long_cuits = set(long["cuit_normalizado"].astype(str))
panel_cuits = set(panel["cuit_normalizado"].astype(str))
enriched_cuits = set(empresas_enriquecida["cuit_normalizado"].astype(str))

coverage_integrity = pd.DataFrame([
    {"control": "Input sin fila de coverage", "cantidad": len(input_cuits - coverage_cuits)},
    {"control": "Coverage con CUIT ajeno al input", "cantidad": len(coverage_cuits - input_cuits)},
    {"control": "Input sin filas long", "cantidad": len(input_cuits - long_cuits)},
    {"control": "Input sin filas empresa-mes", "cantidad": len(input_cuits - panel_cuits)},
    {"control": "Input ausente del maestro enriquecido", "cantidad": len(input_cuits - enriched_cuits)},
    {"control": "CUIT nuevo en long", "cantidad": len(long_cuits - input_cuits)},
    {"control": "CUIT nuevo en panel", "cantidad": len(panel_cuits - input_cuits)},
])
display(coverage_integrity)


,attempted,http_status,outcome,empresas
0,True,200,ok,100


,control,cantidad
0,Input sin fila de coverage,0
1,Coverage con CUIT ajeno al input,0
2,Input sin filas long,0
3,Input sin filas empresa-mes,0
4,Input ausente del maestro enriquecido,0
5,CUIT nuevo en long,0
6,CUIT nuevo en panel,0


In [12]:
# HTTP 200 sin filas históricas: distinguir respuesta exitosa de histórico utilizable.
cov_tmp = coverage.copy()
cov_tmp["cuit_normalizado"] = cov_tmp["cuit_normalizado"].astype(str)
long_counts = long.groupby("cuit_normalizado").size().rename("filas_long").reset_index()
long_counts["cuit_normalizado"] = long_counts["cuit_normalizado"].astype(str)

cov_history = cov_tmp.merge(long_counts, on="cuit_normalizado", how="left")
cov_history["filas_long"] = cov_history["filas_long"].fillna(0).astype(int)

ok_zero_history = cov_history[
    (cov_history["outcome"].eq("ok")) & (cov_history["filas_long"].eq(0))
]
print("HTTP 200 / ok con cero filas históricas:", len(ok_zero_history))
display(ok_zero_history.head(20))


HTTP 200 / ok con cero filas históricas: 0


,empresa_id,cuit_normalizado,razon_social,rubro,subsectores,attempted,http_status,outcome,message,filas_long


## 4. Calidad estructural

Se revisan duplicados, nulos, CUITs, períodos, entidades, situaciones, montos y flags.  
Los valores `situacion = 0` se **describen**, pero no se corrigen ni se interpretan como error en esta etapa.


In [13]:
# Nulos y filas completamente vacías
null_rows = []
for name, df in tables.items():
    for col in df.columns:
        null_rows.append({
            "tabla": name,
            "columna": col,
            "nulos": int(df[col].isna().sum()),
            "pct_nulos": float(df[col].isna().mean()),
            "vacios_string": int(df[col].astype(str).str.strip().eq("").sum()),
        })

null_summary = pd.DataFrame(null_rows)
display(
    null_summary[(null_summary["nulos"] > 0) | (null_summary["vacios_string"] > 0)]
    .sort_values(["tabla", "pct_nulos"], ascending=[True, False])
)

empty_row_summary = pd.DataFrame([
    {
        "tabla": name,
        "filas_completamente_vacias": int(
            df.replace(r"^\s*$", np.nan, regex=True).isna().all(axis=1).sum()
        )
    }
    for name, df in tables.items()
])
display(empty_row_summary)


,tabla,columna,nulos,pct_nulos,vacios_string
22,coverage,message,100,1.000000,0
30,empresa_entidad_mes,grupo_economico,26424,0.912179,0
44,empresa_mes,grupo_economico,2208,0.920000,0
61,empresa_mes,deuda_var_pct_1m,147,0.061250,0
59,empresa_mes,deuda_lag_1m,100,0.041667,0
60,empresa_mes,deuda_var_abs_1m_miles,100,0.041667,0
62,empresa_mes,n_entidades_lag_1m,100,0.041667,0
63,empresa_mes,n_entidades_delta_1m,100,0.041667,0
64,empresa_mes,situacion_lag_1m,100,0.041667,0
65,empresa_mes,situacion_delta_1m,100,0.041667,0


,tabla,filas_completamente_vacias
0,empresas_input,0
1,coverage,0
2,empresa_entidad_mes,0
3,empresa_mes,0
4,empresa_subsector_input,0
5,empresa_subsector_mes,0
6,empresas_enriquecida,0


In [14]:
# Validación de CUIT argentino
def normalize_digits(x):
    return re.sub(r"\D", "", str(x))

def valid_cuit_check_digit(cuit):
    cuit = normalize_digits(cuit)
    if len(cuit) != 11 or not cuit.isdigit():
        return False
    nums = [int(x) for x in cuit]
    weights = [5, 4, 3, 2, 7, 6, 5, 4, 3, 2]
    total = sum(n * w for n, w in zip(nums[:10], weights))
    r = 11 - (total % 11)
    check = 0 if r == 11 else 9 if r == 10 else r
    return check == nums[10]

cuit_diag = empresas[["empresa_id", "razon_social", "cuit_normalizado"]].copy()
cuit_diag["formato_11_digitos"] = cuit_diag["cuit_normalizado"].map(
    lambda x: len(normalize_digits(x)) == 11
)
cuit_diag["digito_verificador_valido"] = cuit_diag["cuit_normalizado"].map(
    valid_cuit_check_digit
)

display(pd.DataFrame([{
    "empresas": len(cuit_diag),
    "formato_invalido": int((~cuit_diag["formato_11_digitos"]).sum()),
    "digito_verificador_invalido": int((~cuit_diag["digito_verificador_valido"]).sum()),
    "cuit_duplicados": int(empresas["cuit_normalizado"].duplicated(keep=False).sum()),
}]))
display(cuit_diag[
    (~cuit_diag["formato_11_digitos"]) |
    (~cuit_diag["digito_verificador_valido"])
].head(30))


,empresas,formato_invalido,digito_verificador_invalido,cuit_duplicados
0,100,0,0,0


,empresa_id,razon_social,cuit_normalizado,formato_11_digitos,digito_verificador_valido


In [15]:
# Períodos
periodo_str = long["periodo"].astype(str)
periodo_valid_format = periodo_str.str.fullmatch(r"\d{6}", na=False)
periodo_parsed = pd.to_datetime(periodo_str + "01", format="%Y%m%d", errors="coerce")

period_check = pd.DataFrame([{
    "filas_long": len(long),
    "periodo_formato_invalido": int((~periodo_valid_format).sum()),
    "periodo_fecha_invalida": int(periodo_parsed.isna().sum()),
    "primera_fecha": periodo_parsed.min(),
    "ultima_fecha": periodo_parsed.max(),
}])
display(period_check)

# Entidades
entity_diag = pd.DataFrame([{
    "entidad_nula": int(long["entidad"].isna().sum()),
    "entidad_vacia": int(long["entidad"].astype(str).str.strip().eq("").sum()),
    "espacios_extremos": int(
        (long["entidad"].astype(str) != long["entidad"].astype(str).str.strip()).sum()
    ),
    "duplicados_clave_cuit_periodo_entidad": int(
        long.duplicated(["cuit_normalizado", "periodo", "entidad"], keep=False).sum()
    ),
}])
display(entity_diag)

# Colisiones simples tras normalizar texto de entidad
entity_variants = long[["entidad"]].drop_duplicates().copy()
entity_variants["entidad_norm"] = (
    entity_variants["entidad"].astype(str)
    .str.strip().str.casefold()
    .str.replace(r"\s+", " ", regex=True)
)
collisions = (
    entity_variants.groupby("entidad_norm")["entidad"].nunique()
    .loc[lambda s: s > 1]
)
print("Nombres normalizados con más de una variante textual:", len(collisions))
if len(collisions):
    display(entity_variants[entity_variants["entidad_norm"].isin(collisions.index)].sort_values("entidad_norm"))


,filas_long,periodo_formato_invalido,periodo_fecha_invalida,primera_fecha,ultima_fecha
0,28968,0,0,2024-08-01,2026-07-01


,entidad_nula,entidad_vacia,espacios_extremos,duplicados_clave_cuit_periodo_entidad
0,0,0,0,0


Nombres normalizados con más de una variante textual: 0


In [16]:
# Situaciones, montos y flags
situacion_counts = (
    long_num["situacion"].value_counts(dropna=False)
    .sort_index()
    .rename_axis("situacion")
    .reset_index(name="filas")
)
display(situacion_counts)

amount_diag = pd.DataFrame([{
    "monto_no_numerico_o_nulo": int(long_num["monto_miles_pesos"].isna().sum()),
    "monto_negativo": int((long_num["monto_miles_pesos"] < 0).sum()),
    "monto_cero": int((long_num["monto_miles_pesos"] == 0).sum()),
    "monto_positivo": int((long_num["monto_miles_pesos"] > 0).sum()),
}])
display(amount_diag)

# No se considera '0' error aquí; sólo se identifican valores fuera del conjunto observado esperado.
situacion_fuera_0_5 = long_num[
    long_num["situacion"].notna() &
    ~long_num["situacion"].isin([0, 1, 2, 3, 4, 5])
]
print("Situaciones fuera de {0,1,2,3,4,5}:", len(situacion_fuera_0_5))

for flag in ["en_revision", "proceso_judicial"]:
    vals = long[flag].value_counts(dropna=False)
    print(f"\n{flag}:")
    display(vals.to_frame("filas"))


,situacion,filas
0,0,10687
1,1,17908
2,2,216
3,3,70
4,4,21
5,5,66


,monto_no_numerico_o_nulo,monto_negativo,monto_cero,monto_positivo
0,0,0,10687,18281


Situaciones fuera de {0,1,2,3,4,5}: 0

en_revision:


,filas
en_revision,
False,28968



proceso_judicial:


,filas
proceso_judicial,
False,28968


## 5. Reconciliación `raw JSON → empresa-entidad-mes`

Se reconstruyen las observaciones directamente desde los JSON originales del run y se comparan con `base_c_empresa_entidad_mes.csv`.

Esto valida la transformación; **no evalúa la veracidad del dato reportado por la fuente**.


In [17]:
raw_members = sorted([
    m for m in members
    if m.startswith(f"{RUN_PREFIX}/raw/deudas/") and m.endswith(".json")
])

raw_rows = []
for member in raw_members:
    cuit = Path(member).stem
    payload = read_json_z(member)
    results = payload.get("results") or {}
    denominacion = results.get("denominacion")
    for p in results.get("periodos") or []:
        periodo = str(p.get("periodo", ""))
        for e in p.get("entidades") or []:
            raw_rows.append({
                "cuit_normalizado": cuit,
                "periodo": periodo,
                "entidad": e.get("entidad"),
                "denominacion_bcra": denominacion,
                "situacion": e.get("situacion"),
                "monto_miles_pesos": e.get("monto"),
                "en_revision": e.get("enRevision"),
                "proceso_judicial": e.get("procesoJud"),
            })

raw_flat = pd.DataFrame(raw_rows)

for df in [raw_flat, long_num]:
    df["cuit_normalizado"] = df["cuit_normalizado"].astype(str)
    df["periodo"] = df["periodo"].astype(str)

raw_flat["situacion"] = pd.to_numeric(raw_flat["situacion"], errors="coerce")
raw_flat["monto_miles_pesos"] = pd.to_numeric(raw_flat["monto_miles_pesos"], errors="coerce")

key_long = ["cuit_normalizado", "periodo", "entidad"]
raw_keys = raw_flat[key_long].drop_duplicates()
long_keys = long_num[key_long].drop_duplicates()

key_merge = raw_keys.merge(long_keys, on=key_long, how="outer", indicator=True)

display(pd.DataFrame([{
    "raw_json": len(raw_members),
    "filas_reconstruidas_raw": len(raw_flat),
    "filas_long": len(long),
    "claves_solo_raw": int((key_merge["_merge"] == "left_only").sum()),
    "claves_solo_long": int((key_merge["_merge"] == "right_only").sum()),
}]))

# Comparación de valores en claves comunes
compare_cols = [
    "situacion", "monto_miles_pesos",
    "en_revision", "proceso_judicial", "denominacion_bcra"
]
cmp = raw_flat.merge(
    long_num[key_long + compare_cols],
    on=key_long,
    how="inner",
    suffixes=("_raw", "_long"),
)

def normalize_compare_series(s, field):
    if field in ["en_revision", "proceso_judicial"]:
        return (
            s.astype(str).str.strip().str.casefold()
            .replace({"true": "true", "false": "false", "1": "true", "0": "false"})
        )
    if field in ["situacion", "monto_miles_pesos"]:
        return pd.to_numeric(s, errors="coerce")
    return s.astype("string").str.strip()

mismatch_rows = []
for c in compare_cols:
    a = normalize_compare_series(cmp[f"{c}_raw"], c)
    b = normalize_compare_series(cmp[f"{c}_long"], c)
    same = (a.eq(b)) | (a.isna() & b.isna())
    mismatch_rows.append({
        "campo": c,
        "mismatches": int((~same).sum())
    })
display(pd.DataFrame(mismatch_rows))


,raw_json,filas_reconstruidas_raw,filas_long,claves_solo_raw,claves_solo_long
0,100,28968,28968,0,0


,campo,mismatches
0,situacion,0
1,monto_miles_pesos,0
2,en_revision,0
3,proceso_judicial,0
4,denominacion_bcra,0


## 6. Cobertura temporal

Se mide primera/última fecha, cantidad de meses y gaps internos por empresa.

**Importante:** si existiera un gap, este notebook **no lo convierte en deuda cero**. Un gap debe investigarse como posible ausencia de información antes de interpretarlo.


In [18]:
def month_range(start, end):
    if pd.isna(start) or pd.isna(end):
        return pd.DatetimeIndex([])
    return pd.date_range(start=start, end=end, freq="MS")

temporal_rows = []
for cuit, g in panel_num.groupby("cuit_normalizado"):
    dates = pd.DatetimeIndex(sorted(g["periodo_dt_eda"].dropna().unique()))
    first = dates.min() if len(dates) else pd.NaT
    last = dates.max() if len(dates) else pd.NaT
    expected = month_range(first, last)
    missing = expected.difference(dates)
    temporal_rows.append({
        "cuit_normalizado": str(cuit),
        "primera_fecha": first,
        "ultima_fecha": last,
        "meses_observados": len(dates),
        "meses_esperados_entre_extremos": len(expected),
        "gaps_internos": len(missing),
        "meses_faltantes": ", ".join(d.strftime("%Y-%m") for d in missing[:12]),
    })

temporal = pd.DataFrame(temporal_rows)

display(temporal.describe(include="all"))
display(
    temporal.groupby(
        ["primera_fecha", "ultima_fecha", "meses_observados", "gaps_internos"],
        dropna=False
    ).size().reset_index(name="empresas").sort_values("empresas", ascending=False)
)

print("Empresas con gaps internos:", int((temporal["gaps_internos"] > 0).sum()))
display(temporal[temporal["gaps_internos"] > 0].head(30))


,cuit_normalizado,primera_fecha,ultima_fecha,meses_observados,meses_esperados_entre_extremos,gaps_internos,meses_faltantes
count,100,100,100,100.0,100.0,100.0,100
unique,100,NaN,NaN,NaN,NaN,NaN,1
top,30500120882,NaN,NaN,NaN,NaN,NaN,
freq,1,NaN,NaN,NaN,NaN,NaN,100
mean,NaN,2024-08-01 00:00:00,2026-07-01 00:00:00,24.0,24.0,0.0,NaN
min,NaN,2024-08-01 00:00:00,2026-07-01 00:00:00,24.0,24.0,0.0,NaN
25%,NaN,2024-08-01 00:00:00,2026-07-01 00:00:00,24.0,24.0,0.0,NaN
50%,NaN,2024-08-01 00:00:00,2026-07-01 00:00:00,24.0,24.0,0.0,NaN
75%,NaN,2024-08-01 00:00:00,2026-07-01 00:00:00,24.0,24.0,0.0,NaN
max,NaN,2024-08-01 00:00:00,2026-07-01 00:00:00,24.0,24.0,0.0,NaN


,primera_fecha,ultima_fecha,meses_observados,gaps_internos,empresas
0,2024-08-01,2026-07-01,24,0,100


Empresas con gaps internos: 0


,cuit_normalizado,primera_fecha,ultima_fecha,meses_observados,meses_esperados_entre_extremos,gaps_internos,meses_faltantes


## 7. Consistencia `empresa-entidad-mes → empresa-mes`

Se recalculan las agregaciones principales desde el long y se comparan con el panel almacenado.

No se modifican las definiciones actuales.


In [19]:
def hhi_amounts(values):
    x = pd.to_numeric(values, errors="coerce").dropna()
    x = x[x > 0]
    total = x.sum()
    if total <= 0:
        return np.nan
    shares = x / total
    return float((shares ** 2).sum())

def top_share(values, n):
    x = pd.to_numeric(values, errors="coerce").dropna()
    x = x[x > 0]
    total = x.sum()
    if total <= 0:
        return np.nan
    return float(x.nlargest(n).sum() / total)

def boolish(s):
    return (
        s.astype(str).str.strip().str.casefold()
        .map({"true": True, "false": False, "1": True, "0": False})
    )

long_calc = long_num.copy()
long_calc["en_revision_bool"] = boolish(long_calc["en_revision"])
long_calc["proceso_judicial_bool"] = boolish(long_calc["proceso_judicial"])

# Compatible con distintas versiones de pandas: groupby + loop explícito.
agg_rows = []
for (cuit, periodo), g in long_calc.groupby(["cuit_normalizado", "periodo"], dropna=False):
    agg_rows.append({
        "cuit_normalizado": str(cuit),
        "periodo": str(periodo),
        "deuda_total_miles": g["monto_miles_pesos"].sum(min_count=1),
        "deuda_promedio_entidad_miles": g["monto_miles_pesos"].mean(),
        "deuda_mediana_entidad_miles": g["monto_miles_pesos"].median(),
        "deuda_max_entidad_miles": g["monto_miles_pesos"].max(),
        "n_entidades": g["entidad"].nunique(dropna=True),
        "situacion_max": g["situacion"].max(),
        "situacion_promedio": g["situacion"].mean(),
        "en_revision_any": bool(g["en_revision_bool"].fillna(False).any()),
        "proceso_judicial_any": bool(g["proceso_judicial_bool"].fillna(False).any()),
        "hhi_concentracion_deuda": hhi_amounts(g["monto_miles_pesos"]),
        "top1_share_deuda": top_share(g["monto_miles_pesos"], 1),
        "top3_share_deuda": top_share(g["monto_miles_pesos"], 3),
    })
agg = pd.DataFrame(agg_rows)

panel_cmp = panel_num.copy()
panel_cmp["cuit_normalizado"] = panel_cmp["cuit_normalizado"].astype(str)
agg["cuit_normalizado"] = agg["cuit_normalizado"].astype(str)

keys = ["cuit_normalizado", "periodo"]
key_cmp = agg[keys].merge(panel_cmp[keys], on=keys, how="outer", indicator=True)

display(pd.DataFrame([{
    "empresa_mes_recalculadas": len(agg),
    "empresa_mes_archivo": len(panel_cmp),
    "claves_solo_recalculadas": int((key_cmp["_merge"] == "left_only").sum()),
    "claves_solo_panel": int((key_cmp["_merge"] == "right_only").sum()),
}]))

compare = agg.merge(panel_cmp, on=keys, how="inner", suffixes=("_calc", "_panel"))

numeric_compare = [
    "deuda_total_miles",
    "deuda_promedio_entidad_miles",
    "deuda_mediana_entidad_miles",
    "deuda_max_entidad_miles",
    "n_entidades",
    "situacion_max",
    "situacion_promedio",
    "hhi_concentracion_deuda",
    "top1_share_deuda",
    "top3_share_deuda",
]

agg_mismatch = []
for c in numeric_compare:
    a = pd.to_numeric(compare[f"{c}_calc"], errors="coerce")
    b = pd.to_numeric(compare[f"{c}_panel"], errors="coerce")
    same = np.isclose(a, b, equal_nan=True, rtol=1e-10, atol=1e-10)
    agg_mismatch.append({"campo": c, "mismatches": int((~same).sum())})

for c in ["en_revision_any", "proceso_judicial_any"]:
    a = compare[f"{c}_calc"].astype(str).str.casefold()
    b = compare[f"{c}_panel"].astype(str).str.casefold()
    agg_mismatch.append({"campo": c, "mismatches": int((a != b).sum())})

display(pd.DataFrame(agg_mismatch))


,empresa_mes_recalculadas,empresa_mes_archivo,claves_solo_recalculadas,claves_solo_panel
0,2400,2400,0,0


,campo,mismatches
0,deuda_total_miles,0
1,deuda_promedio_entidad_miles,0
2,deuda_mediana_entidad_miles,0
3,deuda_max_entidad_miles,0
4,n_entidades,0
5,situacion_max,0
6,situacion_promedio,0
7,hhi_concentracion_deuda,0
8,top1_share_deuda,0
9,top3_share_deuda,0


## 8. Diagnóstico de `monto = 0` y `n_entidades`

**No se cambia `n_entidades`.**

La variable actual cuenta **entidades presentes en el conjunto de filas BCRA del empresa-mes**, incluso cuando `monto = 0`.

Como control de calidad se calcula, sólo para comparar:

- `n_entidades_actual`: definición vigente;
- `n_entidades_monto_positivo`: cantidad de entidades con `monto > 0`;
- diferencia entre ambas.

Esto se documenta como **DECISIÓN METODOLÓGICA**, no como corrección.


In [20]:
long_zero = long_num.copy()
long_zero["monto_es_cero"] = long_zero["monto_miles_pesos"].eq(0)
long_zero["monto_es_positivo"] = long_zero["monto_miles_pesos"].gt(0)

zero_summary = pd.DataFrame([{
    "filas_long": len(long_zero),
    "filas_monto_cero": int(long_zero["monto_es_cero"].sum()),
    "pct_monto_cero": float(long_zero["monto_es_cero"].mean()),
    "empresas_con_algun_monto_cero": int(
        long_zero.loc[long_zero["monto_es_cero"], "cuit_normalizado"].nunique()
    ),
}])
display(zero_summary)

print("Situación × monto cero:")
display(pd.crosstab(
    long_zero["situacion"],
    long_zero["monto_es_cero"],
    margins=True
))

entity_rows = []
for (cuit, periodo), g in long_zero.groupby(["cuit_normalizado", "periodo"], dropna=False):
    entity_rows.append({
        "cuit_normalizado": str(cuit),
        "periodo": str(periodo),
        "n_entidades_actual": g["entidad"].nunique(dropna=True),
        "n_entidades_monto_positivo": g.loc[
            g["monto_miles_pesos"] > 0, "entidad"
        ].nunique(dropna=True),
    })
entity_month = pd.DataFrame(entity_rows)
entity_month["diferencia"] = (
    entity_month["n_entidades_actual"] -
    entity_month["n_entidades_monto_positivo"]
)

display(pd.DataFrame([{
    "empresa_mes": len(entity_month),
    "empresa_mes_afectados": int((entity_month["diferencia"] > 0).sum()),
    "pct_empresa_mes_afectados": float((entity_month["diferencia"] > 0).mean()),
    "empresas_afectadas": int(
        entity_month.loc[entity_month["diferencia"] > 0, "cuit_normalizado"].nunique()
    ),
}]))

display(
    entity_month["diferencia"].value_counts()
    .sort_index()
    .rename_axis("diferencia")
    .reset_index(name="empresa_mes")
)

# ¿n_entidades actual es constante en el tiempo?
constancy = (
    entity_month.groupby("cuit_normalizado")
    .agg(
        valores_distintos_n_entidades_actual=("n_entidades_actual", "nunique"),
        valores_distintos_n_entidades_positivas=("n_entidades_monto_positivo", "nunique"),
    )
    .reset_index()
)
display(pd.DataFrame([{
    "empresas_n_entidades_actual_constante": int(
        (constancy["valores_distintos_n_entidades_actual"] == 1).sum()
    ),
    "empresas_n_entidades_positivas_variable": int(
        (constancy["valores_distintos_n_entidades_positivas"] > 1).sum()
    ),
}]))


,filas_long,filas_monto_cero,pct_monto_cero,empresas_con_algun_monto_cero
0,28968,10687,0.368924,98


Situación × monto cero:


monto_es_cero,False,True,All
situacion,,,
0,0,10687,10687
1,17908,0,17908
2,216,0,216
3,70,0,70
4,21,0,21
5,66,0,66
All,18281,10687,28968


,empresa_mes,empresa_mes_afectados,pct_empresa_mes_afectados,empresas_afectadas
0,2400,2202,0.9175,98


,diferencia,empresa_mes
0,0,198
1,1,266
2,2,307
3,3,333
4,4,277
5,5,261
6,6,234
7,7,127
8,8,93
9,9,89


,empresas_n_entidades_actual_constante,empresas_n_entidades_positivas_variable
0,100,98


### Interpretación prudente

En este EDA:

- `monto > 0` = saldo positivo informado en Central de Deudores para esa fila;
- `monto = 0` = entidad presente sin saldo positivo informado en esa fila;
- `monto = 0` **no prueba por sí mismo que la empresa no tenga ningún pasivo**;
- Base C observa financiamiento reportado a Central de Deudores, **no el pasivo total de la empresa**;
- no se redefine `n_entidades` en esta etapa.


## 9. Consistencia con la muestra empresarial

Se comprueba que la respuesta crediticia no haya cambiado retrospectivamente la muestra.


In [21]:
sample_checks = pd.DataFrame([
    {
        "control": "CUIT seleccionados input",
        "cantidad": len(input_cuits),
    },
    {
        "control": "CUIT en coverage",
        "cantidad": len(coverage_cuits),
    },
    {
        "control": "CUIT en long",
        "cantidad": len(long_cuits),
    },
    {
        "control": "CUIT en empresa-mes",
        "cantidad": len(panel_cuits),
    },
    {
        "control": "CUIT en maestro enriquecido",
        "cantidad": len(enriched_cuits),
    },
    {
        "control": "CUIT del input ausentes del enriquecido",
        "cantidad": len(input_cuits - enriched_cuits),
    },
    {
        "control": "CUIT nuevos introducidos por BCRA",
        "cantidad": len((coverage_cuits | long_cuits | panel_cuits) - input_cuits),
    },
])
display(sample_checks)

# Relación empresa-subsector
selected_pairs = set(map(tuple, empresa_subsector[
    ["cuit_normalizado", "subsector"]
].astype(str).drop_duplicates().to_numpy()))

panel_pairs = set(map(tuple, panel_subsector[
    ["cuit_normalizado", "subsector"]
].astype(str).drop_duplicates().to_numpy()))

subsector_checks = pd.DataFrame([{
    "asignaciones_input": len(selected_pairs),
    "asignaciones_presentes_panel_subsector": len(panel_pairs),
    "asignaciones_input_faltantes": len(selected_pairs - panel_pairs),
    "asignaciones_nuevas": len(panel_pairs - selected_pairs),
}])
display(subsector_checks)

display(
    empresa_subsector.groupby("subsector")["cuit_normalizado"]
    .nunique()
    .sort_values(ascending=False)
    .rename("empresas")
    .reset_index()
)


,control,cantidad
0,CUIT seleccionados input,100
1,CUIT en coverage,100
2,CUIT en long,100
3,CUIT en empresa-mes,100
4,CUIT en maestro enriquecido,100
5,CUIT del input ausentes del enriquecido,0
6,CUIT nuevos introducidos por BCRA,0


,asignaciones_input,asignaciones_presentes_panel_subsector,asignaciones_input_faltantes,asignaciones_nuevas
0,102,102,0,0


,subsector,empresas
0,Granos/Oleaginosas,19
1,Carne bovina,13
2,Lácteos,8
3,Frutas y citrus,7
4,Avícola,7
5,Foresto-industria,6
6,Molinería/Transformación cerealera,6
7,Azúcar/Bioetanol,6
8,Vitivinicultura,6
9,Maní,5


## 10. Target exploratorio — diferido

El archivo `empresa_mes` puede contener columnas derivadas relacionadas con un target exploratorio.

**En este notebook:**
- no se redefine;
- no se optimiza;
- no se juzga su prevalencia;
- no se corrigen transiciones;
- no se usa para decidir calidad de la muestra.

La discusión de target, leakage, elegibilidad y definición temporal se hará **post-EDA**, al comenzar el diseño del modelo.


In [22]:
target_cols = [
    c for c in panel.columns
    if "target" in c.lower() or "next" in c.lower()
]
print("Columnas relacionadas presentes (sólo inventario):", target_cols)


Columnas relacionadas presentes (sólo inventario): ['situacion_next_1m', 'target_deteriora_1m_exploratorio']


## 11. Matriz final de preguntas de calidad

La siguiente tabla resume preguntas a **considerar/descartar** sin modificar los datos.


In [23]:
# Variables auxiliares para estado de checks
dup_long = int(long.duplicated(["cuit_normalizado", "periodo", "entidad"], keep=False).sum())
dup_panel = int(panel.duplicated(["cuit_normalizado", "periodo"], keep=False).sum())
invalid_cuit = int((~cuit_diag["digito_verificador_valido"]).sum())
invalid_period = int(periodo_parsed.isna().sum())
negative_amounts = int((long_num["monto_miles_pesos"] < 0).sum())
outside_situation = len(situacion_fuera_0_5)
gaps_companies = int((temporal["gaps_internos"] > 0).sum())
raw_key_missing = int((key_merge["_merge"] != "both").sum())
agg_key_missing = int((key_cmp["_merge"] != "both").sum())
agg_value_mismatch = int(pd.DataFrame(agg_mismatch)["mismatches"].sum())
sample_new = len((coverage_cuits | long_cuits | panel_cuits) - input_cuits)
sample_missing_enriched = len(input_cuits - enriched_cuits)
subsector_missing = len(selected_pairs - panel_pairs)
subsector_new = len(panel_pairs - selected_pairs)

def status(condition, pass_text="PASS", fail_text="REVISAR"):
    return pass_text if condition else fail_text

questions = pd.DataFrame([
    ["Q01", "Inventario", "¿Hay un run completo identificable y trazable?", "PASS", "DECISIÓN METODOLÓGICA",
     RUN_ID_SELECTED],
    ["Q02", "Cobertura", "¿Coverage tiene exactamente una fila por empresa input?",
     status(len(coverage)==len(empresas) and coverage["cuit_normalizado"].nunique()==len(empresas)),
     "ERROR DE DATOS", f"{len(coverage)} coverage / {len(empresas)} input"],
    ["Q03", "Cobertura", "¿Todos los CUIT consultados pertenecen al maestro?",
     status(len(coverage_cuits-input_cuits)==0), "ERROR DE DATOS",
     f"CUIT ajenos: {len(coverage_cuits-input_cuits)}"],
    ["Q04", "Muestra", "¿Todas las empresas input permanecen en el maestro enriquecido?",
     status(sample_missing_enriched==0), "ERROR DE DATOS",
     f"Faltantes: {sample_missing_enriched}"],
    ["Q05", "Raw", "¿El raw se reconcilia con empresa-entidad-mes?",
     status(raw_key_missing==0 and all(x['mismatches']==0 for x in mismatch_rows)),
     "ERROR DE DATOS", f"Diferencias de clave: {raw_key_missing}"],
    ["Q06", "Cobertura", "¿Existen HTTP 200 sin filas históricas?",
     "REVISAR" if len(ok_zero_history) else "PASS", "AUSENCIA DE INFORMACIÓN",
     f"Casos: {len(ok_zero_history)}"],
    ["Q07", "Cobertura", "¿Un 404 puede interpretarse como deuda cero?",
     "NO: NO ASUMIR", "DECISIÓN METODOLÓGICA",
     "404 se conserva como ausencia de información"],
    ["Q08", "CUIT", "¿Todos los CUIT tienen dígito verificador válido?",
     status(invalid_cuit==0), "ERROR DE DATOS", f"Inválidos: {invalid_cuit}"],
    ["Q09", "Claves", "¿Hay duplicados CUIT+periodo+entidad?",
     status(dup_long==0), "ERROR DE DATOS", f"Filas duplicadas: {dup_long}"],
    ["Q10", "Claves", "¿Hay duplicados CUIT+periodo en empresa-mes?",
     status(dup_panel==0), "ERROR DE DATOS", f"Filas duplicadas: {dup_panel}"],
    ["Q11", "Período", "¿Todos los períodos son fechas mensuales válidas?",
     status(invalid_period==0), "ERROR DE DATOS", f"Inválidos: {invalid_period}"],
    ["Q12", "Monto", "¿Existen montos negativos?",
     status(negative_amounts==0), "ERROR DE DATOS", f"Negativos: {negative_amounts}"],
    ["Q13", "Situación", "¿Existen situaciones fuera de 0-5?",
     status(outside_situation==0), "ERROR DE DATOS", f"Fuera de 0-5: {outside_situation}"],
    ["Q14", "Temporal", "¿Existen gaps internos?",
     "REVISAR" if gaps_companies else "PASS", "AUSENCIA DE INFORMACIÓN",
     f"Empresas con gaps: {gaps_companies}"],
    ["Q15", "Agregación", "¿Todas las claves empresa-mes se preservan al agregar?",
     status(agg_key_missing==0), "ERROR DE DATOS", f"Diferencias: {agg_key_missing}"],
    ["Q16", "Agregación", "¿Las métricas empresa-mes se reproducen desde el long?",
     status(agg_value_mismatch==0), "ERROR DE DATOS", f"Mismatches: {agg_value_mismatch}"],
    ["Q17", "Muestra", "¿Aparecen CUIT nuevos por la respuesta BCRA?",
     status(sample_new==0), "ERROR DE DATOS", f"Nuevos: {sample_new}"],
    ["Q18", "Subsector", "¿Se preservan las asignaciones empresa-subsector?",
     status(subsector_missing==0 and subsector_new==0), "ERROR DE DATOS",
     f"Faltantes: {subsector_missing}; nuevas: {subsector_new}"],
    ["Q19", "Monto 0", "¿Monto 0 afecta el conteo actual de n_entidades?",
     "DOCUMENTADO", "DECISIÓN METODOLÓGICA",
     f"Empresa-mes afectados: {(entity_month['diferencia']>0).sum()}"],
    ["Q20", "Target", "Definición, leakage y calidad del target",
     "POST-EDA", "DECISIÓN METODOLÓGICA",
     "Diferido hasta diseño del modelo"],
], columns=["id", "tema", "pregunta", "estado", "categoria", "evidencia"])

display(questions)


,id,tema,pregunta,estado,categoria,evidencia
0,Q01,Inventario,¿Hay un run completo identificable y trazable?,PASS,DECISIÓN METODOLÓGICA,20260910_234243_324893
1,Q02,Cobertura,¿Coverage tiene exactamente una fila por empre...,PASS,ERROR DE DATOS,100 coverage / 100 input
2,Q03,Cobertura,¿Todos los CUIT consultados pertenecen al maes...,PASS,ERROR DE DATOS,CUIT ajenos: 0
3,Q04,Muestra,¿Todas las empresas input permanecen en el mae...,PASS,ERROR DE DATOS,Faltantes: 0
4,Q05,Raw,¿El raw se reconcilia con empresa-entidad-mes?,PASS,ERROR DE DATOS,Diferencias de clave: 0
5,Q06,Cobertura,¿Existen HTTP 200 sin filas históricas?,PASS,AUSENCIA DE INFORMACIÓN,Casos: 0
6,Q07,Cobertura,¿Un 404 puede interpretarse como deuda cero?,NO: NO ASUMIR,DECISIÓN METODOLÓGICA,404 se conserva como ausencia de información
7,Q08,CUIT,¿Todos los CUIT tienen dígito verificador válido?,PASS,ERROR DE DATOS,Inválidos: 0
8,Q09,Claves,¿Hay duplicados CUIT+periodo+entidad?,PASS,ERROR DE DATOS,Filas duplicadas: 0
9,Q10,Claves,¿Hay duplicados CUIT+periodo en empresa-mes?,PASS,ERROR DE DATOS,Filas duplicadas: 0


## 12. Resumen automático

Este bloque sintetiza los resultados de calidad sin modificar Base C.


In [24]:
outcomes = coverage["outcome"].value_counts(dropna=False).to_dict()
months_median = temporal["meses_observados"].median() if len(temporal) else np.nan

summary_lines = [
    f"### Resumen del run `{RUN_ID_SELECTED}`",
    "",
    f"- Empresas input: **{len(empresas):,}**",
    f"- Consultas registradas: **{len(coverage):,}**",
    f"- HTTP/outcome `ok`: **{int((coverage['outcome']=='ok').sum()):,}**",
    f"- Filas empresa-entidad-mes: **{len(long):,}**",
    f"- Filas empresa-mes: **{len(panel):,}**",
    f"- Empresas presentes en empresa-mes: **{panel['cuit_normalizado'].nunique():,}**",
    f"- Mediana de meses por empresa: **{months_median:.1f}**",
    f"- Empresas con gaps internos: **{gaps_companies:,}**",
    f"- Duplicados de clave long: **{dup_long:,}**",
    f"- Duplicados de clave panel: **{dup_panel:,}**",
    f"- CUIT con dígito verificador inválido: **{invalid_cuit:,}**",
    f"- Montos negativos: **{negative_amounts:,}**",
    f"- Filas con monto cero: **{int((long_num['monto_miles_pesos']==0).sum()):,}**",
    f"- Diferencias raw → long: **{raw_key_missing + sum(x['mismatches'] for x in mismatch_rows):,}**",
    f"- Mismatches de agregación long → empresa-mes: **{agg_key_missing + agg_value_mismatch:,}**",
    f"- CUIT nuevos introducidos por la respuesta crediticia: **{sample_new:,}**",
    f"- Asignaciones empresa-subsector faltantes/nuevas: **{subsector_missing:,} / {subsector_new:,}**",
    "",
    "**Decisiones mantenidas:** no imputar, no eliminar, no corregir; `n_entidades` se conserva como está y el target se analiza post-EDA.",
]
display(Markdown("\n".join(summary_lines)))


### Resumen del run `20260910_234243_324893`

- Empresas input: **100**
- Consultas registradas: **100**
- HTTP/outcome `ok`: **100**
- Filas empresa-entidad-mes: **28,968**
- Filas empresa-mes: **2,400**
- Empresas presentes en empresa-mes: **100**
- Mediana de meses por empresa: **24.0**
- Empresas con gaps internos: **0**
- Duplicados de clave long: **0**
- Duplicados de clave panel: **0**
- CUIT con dígito verificador inválido: **0**
- Montos negativos: **0**
- Filas con monto cero: **10,687**
- Diferencias raw → long: **0**
- Mismatches de agregación long → empresa-mes: **0**
- CUIT nuevos introducidos por la respuesta crediticia: **0**
- Asignaciones empresa-subsector faltantes/nuevas: **0 / 0**

**Decisiones mantenidas:** no imputar, no eliminar, no corregir; `n_entidades` se conserva como está y el target se analiza post-EDA.